# NB02 — Preprocessing & 5-Class Consolidation

Loads the merged benchmark, applies the (unchanged) cleaning pipeline, **deduplicates on cleaned
text**, then consolidates the 89 raw multi-label variants into the **5-class taxonomy** and saves
`data/processed/benchmark_cleaned.csv` with a final `label5` column.

5 classes: **none · abusive · sexual · religious · threat**  (see plans.md §3).


In [ ]:
import os, re, sys, warnings, unicodedata
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)
try:
    import emoji; HAVE_EMOJI = True
except ImportError:
    HAVE_EMOJI = False; print("emoji lib not found — regex fallback")
print("pandas", pd.__version__)

In [ ]:
# ── Load merged benchmark from NB01 ─────────────────────────────────────────
MERGED = "../data/merged/benchmark_raw.csv"
df = pd.read_csv(MERGED)
print(f"Loaded {MERGED}  shape={df.shape}")
print("Columns:", list(df.columns))
for c in ["text", "label_binary", "label_type", "source", "script"]:
    assert c in df.columns, f"missing column {c}"
print("\nSources:", df["source"].value_counts().to_dict())
print("Scripts:", df["script"].value_counts().to_dict())

In [ ]:
# ── Cleaning pipeline (UNCHANGED from v2) ───────────────────────────────────
URL_RE     = re.compile(r"https?://\S+|www\.\S+|[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}/\S*")
MENTION_RE = re.compile(r"@[\w]+")
HASHTAG_RE = re.compile(r"#(\S+)")
ZW_RE      = re.compile(r"[\u200b\u200c\u200d\ufeff]")
WS_RE      = re.compile(r"\s+")
ELONG_RE   = re.compile(r"(.)\1{2,}")          # collapse 3+ repeats -> 2
EMOJI_RE   = re.compile(
    "[" "\U0001F600-\U0001F64F" "\U0001F300-\U0001F5FF" "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF" "\U00002700-\U000027BF" "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF" "\U0001FA70-\U0001FAFF" "]+", flags=re.UNICODE)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    t = unicodedata.normalize("NFKC", text)
    t = ZW_RE.sub("", t)
    t = URL_RE.sub(" [URL] ", t)
    t = MENTION_RE.sub(" [USER] ", t)
    t = HASHTAG_RE.sub(r" [HASHTAG] \1 ", t)
    if HAVE_EMOJI:
        t = emoji.demojize(t, delimiters=(" [EMOJI_", "] "))
    else:
        t = EMOJI_RE.sub(" [EMOJI] ", t)
    t = ELONG_RE.sub(r"\1\1", t)
    t = WS_RE.sub(" ", t).strip()
    return t

df["text_clean"] = df["text"].apply(preprocess_text)
n0 = len(df)
df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)
print(f"Cleaned. Dropped {n0-len(df)} empty rows → {len(df):,}")
for ex in ["এটা একটা টেস্ট @user123 https://example.com 😂😂😂", "tor matha kharappppp???? 🤬"]:
    print(f"  IN : {ex}\n  OUT: {preprocess_text(ex)}")

In [ ]:
# ── 5-CLASS CONSOLIDATION (NEW) ─────────────────────────────────────────────
# Each raw label is split on "," and "_"; components map to buckets; the final
# class is chosen by priority. Priority keeps explicit-violence & identity abuse
# from being swallowed by the generic "abusive" bucket.
COMPONENT_MAP = {
    "none": "none", "not bully": "none", "notbully": "none",
    "sexual": "sexual", "gender": "sexual",
    "religious": "religious", "religion": "religious",
    "threat": "threat", "calltoviolence": "threat",
    "abusive/violence": "abusive", "abusive": "abusive", "violence": "abusive",
    "personal offense": "abusive", "personal": "abusive", "political": "abusive",
    "troll": "abusive", "slander": "abusive", "spam": "abusive",
    "origin": "abusive", "body shaming": "abusive", "misc": "abusive",
}
PRIORITY = ["threat", "sexual", "religious", "abusive", "none"]   # high → low

def consolidate_5class(raw):
    if not isinstance(raw, str) or not raw.strip():
        return "none"
    parts = [p.strip().lower() for p in re.split(r"[,_]", raw.strip()) if p.strip()]
    buckets = set()
    for p in parts:
        if p in COMPONENT_MAP:
            buckets.add(COMPONENT_MAP[p])
        else:
            for key, val in COMPONENT_MAP.items():      # fuzzy fallback
                if key in p:
                    buckets.add(val); break
    if not buckets:
        return "abusive"                                 # flagged-but-unmapped → conservative
    for cls in PRIORITY:
        if cls in buckets:
            return cls
    return "none"

df["label5"] = df["label_type"].apply(consolidate_5class)
# Enforce consistency with the binary flag
df.loc[df["label_binary"] == 0, "label5"] = "none"
df.loc[(df["label_binary"] == 1) & (df["label5"] == "none"), "label5"] = "abusive"

print("5-class distribution (pre-dedup):")
print(df["label5"].value_counts().to_string())
assert df["label5"].nunique() == 5, "expected exactly 5 classes"

In [ ]:
# ── Deduplication on cleaned text (UNCHANGED policy) ────────────────────────
# Keep one row per unique text_clean. Conflict resolution:
#   label5        → highest-priority class among the duplicates (same PRIORITY)
#   label_binary  → 1 if ANY duplicate is harmful
#   source/script → first occurrence (for traceability)
PRIO_IDX = {c: i for i, c in enumerate(PRIORITY)}   # lower idx = higher priority

def resolve_group(g):
    best = min(g["label5"], key=lambda c: PRIO_IDX.get(c, 99))
    return pd.Series({
        "text": g["text"].iloc[0],
        "text_clean": g["text_clean"].iloc[0],
        "label_binary": int(g["label_binary"].max()),
        "label5": best,
        "label_type": g["label_type"].iloc[0],
        "source": g["source"].iloc[0],
        "script": g["script"].iloc[0],
    })

n_before = len(df)
dups = df["text_clean"].duplicated(keep=False).sum()
print(f"Duplicate rows (cleaned): {dups:,} ({dups/n_before*100:.1f}%)")

# Fast path: rows with a unique text_clean pass through untouched; only collapse the dup groups.
is_dup = df["text_clean"].duplicated(keep=False)
unique_part = df[~is_dup].copy()
dup_part = (df[is_dup].groupby("text_clean", sort=False, group_keys=False)
            .apply(resolve_group).reset_index(drop=True))
keep_cols = ["text", "text_clean", "label_binary", "label5", "label_type", "source", "script"]
df = pd.concat([unique_part[keep_cols], dup_part[keep_cols]], ignore_index=True)
# re-enforce consistency after merge
df.loc[df["label_binary"] == 0, "label5"] = "none"
df.loc[(df["label_binary"] == 1) & (df["label5"] == "none"), "label5"] = "abusive"
df = df.reset_index(drop=True)
print(f"After dedup: {n_before:,} → {len(df):,} (removed {n_before-len(df):,})")
print("\nPost-dedup 5-class distribution:")
print(df["label5"].value_counts().to_string())
print("\nPost-dedup binary:", df["label_binary"].value_counts().to_dict())

In [ ]:
# ── EDA: 5-class × source / script (for the paper) ──────────────────────────
os.makedirs("../outputs", exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 4))
vc = df["label5"].value_counts().reindex(PRIORITY[::-1])
vc.plot.barh(ax=ax, color=sns.color_palette("Set2", 5), edgecolor="black")
for i, v in enumerate(vc.values):
    ax.text(v + len(df)*0.004, i, f"{v:,} ({v/len(df)*100:.1f}%)", va="center", fontsize=9)
ax.set_title("5-Class Distribution (deduplicated)", fontweight="bold"); ax.set_xlabel("Count")
plt.tight_layout(); plt.savefig("../outputs/fig_label5_distribution.png", dpi=150, bbox_inches="tight"); plt.show()

print("\n── label5 × source ──");  print(pd.crosstab(df["label5"], df["source"]).to_string())
print("\n── label5 × script ──");  print(pd.crosstab(df["label5"], df["script"]).to_string())
imb = df["label5"].value_counts()
print(f"\nImbalance ratio (max/min): {imb.max()/imb.min():.1f}:1")

In [ ]:
# ── Save processed benchmark ────────────────────────────────────────────────
os.makedirs("../data/processed", exist_ok=True)
out = "../data/processed/benchmark_cleaned.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
print(f"✅ Saved {out}  shape={df.shape}")
print("Columns:", list(df.columns))
df.sample(min(100, len(df)), random_state=42).to_csv(
    "../data/processed/benchmark_cleaned_sample.csv", index=False, encoding="utf-8-sig")

print("\n" + "="*56)
print("BENCHMARK SUMMARY (for paper)")
print("="*56)
print(f"Total unique samples : {len(df):,}")
print(f"Sources              : {sorted(df['source'].unique())}")
print(f"Scripts              : {df['script'].value_counts().to_dict()}")
print(f"5 classes            : {PRIORITY}")
print(f"Harmful / not        : {df['label_binary'].value_counts().to_dict()}")